# NB33 — USGS Full-Metal MWAS and Abundance-Weighted PGLS

Two analyses not in NB27–NB31:

**Analysis 1 (MWAS)**: Sample-level metagenome-wide association study.  
For each MicrobeAtlas USA soil sample: compute community-weighted KO abundance  
(genus relative abundance × KO presence from genome databases, summed over genera).  
Test each KO against every USGS measured soil metal, controlling for pH/SOC/WTD  
in 9 control combinations. FDR correction per metal × control combo.

**Analysis 2 (PGLS)**: Abundance-weighted genus metal exposure.  
Instead of NB27's centroid approach (median genus lat/lon → nearby USGS),  
compute: Σ_s(rel_abund(g,s) × log1p(metal_s)) / Σ_s(rel_abund(g,s)) per genus.  
Run PGLS of KO density ~ abundance-weighted metal, across 9 control combinations  
and all USGS metals. Compare β stability vs NB27.

**New metals**: All USGS soil elements with ≥5,000 USA observations (ppm), not just the core 7.

**Depends on**: nb25_ko_presence_matrix.parquet, nb02_otu_long_cache.parquet,  
usgs_geochem (local), soilgrids_master (local), water_table_depth (local).

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

import sys, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.spatial import cKDTree
from scipy import stats
from scipy.stats import false_discovery_control
import statsmodels.formula.api as smf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
warnings.filterwarnings('ignore')

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/tools')
from figure_style import apply_style, save, PALETTE, FIGW, ROW_H, grid_h
apply_style()

sys.path.insert(0, '/home/hmacgregor/BERIL-research-observatory/projects/comprehensive_metal_ecology/scripts')
from pgls_utils import load_tree, run_pgls, pgls_results_table

ROOT   = Path('/home/hmacgregor/BERIL-research-observatory')
CME    = ROOT / 'projects/comprehensive_metal_ecology'
BIOIN  = ROOT / 'projects/metal_contamination_bioindicators'
USAENV = ROOT / 'projects/usa_env_bioindicators'
DATA   = CME / 'data'
FIGS   = CME / 'figures'
ENVDBS = Path('/home/hmacgregor/data/envdbs')
USGS_DIR = ENVDBS / 'usgs_geochem'

# Cache paths
USGS_GRID_PATH    = DATA / 'nb33_usgs_full_grid.parquet'
SAMPLE_MASTER_PATH = DATA / 'nb33_sample_master.parquet'
GENUS_METAL_PATH  = DATA / 'nb33_genus_abund_weighted_metals.csv'

# Geographic bounds (USA)
LAT_MIN, LAT_MAX = 24.0, 50.0
LON_MIN, LON_MAX = -126.0, -64.0
MAX_DIST_DEG = 1.5   # match NB27 / NB02
MIN_OBS = 5000       # minimum USA soil observations per element

print('Setup done.')

Setup done.


In [2]:
# ── Build USGS full-metal soil grid (cache to disk) ──────────────────────────
if USGS_GRID_PATH.exists():
    print('Loading cached USGS grid...')
    usgs_grid = pd.read_parquet(USGS_GRID_PATH)
else:
    print('Building USGS full-metal grid from raw geochemistry...')
    geo = pd.read_parquet(USGS_DIR / 'usgs_geochem.parquet',
                          columns=['lab_id', 'latitude', 'longitude', 'primary_class'])
    soil_usa = geo[
        (geo['primary_class'] == 'soil') &
        geo['latitude'].between(LAT_MIN, LAT_MAX) &
        geo['longitude'].between(LON_MIN, LON_MAX)
    ].copy()
    print(f'  USA soil sites: {len(soil_usa):,}')

    print('  Loading tbl_chem (ppm only)...')
    chem = pd.read_parquet(USGS_DIR / 'tbl_chem.parquet',
                           columns=['lab_id', 'species', 'units', 'qualified_value'])
    soil_chem = chem[
        (chem['units'] == 'ppm') &
        (chem['qualified_value'] > 0) &
        chem['lab_id'].isin(soil_usa['lab_id'])
    ].copy()
    del chem

    # Elements with >= MIN_OBS USA soil observations
    elem_cov = soil_chem.groupby('species')['lab_id'].nunique()
    keep_elems = elem_cov[elem_cov >= MIN_OBS].index.tolist()
    print(f'  Elements with ≥{MIN_OBS} obs: {keep_elems}')

    soil_chem = soil_chem[soil_chem['species'].isin(keep_elems)]

    # Pivot to wide: lab_id × element
    # Use median per lab_id × species (some sites have duplicates)
    print('  Pivoting...')
    wide = (soil_chem.groupby(['lab_id', 'species'])['qualified_value']
            .median()
            .unstack(fill_value=np.nan)
            .reset_index())
    wide.columns.name = None
    wide = wide.merge(soil_usa[['lab_id', 'latitude', 'longitude']], on='lab_id', how='left')

    # Aggregate to 0.5° grid
    print('  Building 0.5° grid...')
    wide['lat_grid'] = (wide['latitude'] / 0.5).round() * 0.5
    wide['lon_grid'] = (wide['longitude'] / 0.5).round() * 0.5
    elem_cols = [c for c in wide.columns if c in keep_elems]
    grid = (wide.groupby(['lat_grid', 'lon_grid'])[elem_cols]
            .mean()
            .reset_index()
            .rename(columns={'lat_grid': 'lat', 'lon_grid': 'lon'}))
    # Rename element columns with prefix
    rename_map = {e: f'usgs_{e.lower()}' for e in elem_cols}
    grid.rename(columns=rename_map, inplace=True)
    grid.to_parquet(USGS_GRID_PATH, index=False)
    usgs_grid = grid
    print(f'  Grid: {len(usgs_grid):,} cells, {len(elem_cols)} metals → {USGS_GRID_PATH.name}')

METAL_COLS = [c for c in usgs_grid.columns if c.startswith('usgs_')]
METALS     = [c.replace('usgs_', '') for c in METAL_COLS]
print(f'Grid: {len(usgs_grid):,} cells, {len(METALS)} metals: {METALS}')

Loading cached USGS grid...
Grid: 3,311 cells, 49 metals: ['ag', 'as', 'au', 'b', 'ba', 'be', 'bi', 'cd', 'ce', 'co', 'cr', 'cs', 'cu', 'eu', 'ga', 'ge', 'hf', 'hg', 'in', 'la', 'li', 'lu', 'mo', 'nb', 'nd', 'ni', 'pb', 'pd', 'pt', 'rb', 're', 'sb', 'sc', 'se', 'sm', 'sn', 'sr', 'ta', 'tb', 'te', 'th', 'tl', 'u', 'v', 'w', 'y', 'yb', 'zn', 'zr']


In [3]:
# ── Load environmental grids: SoilGrids pH/SOC + WTD + WorldClim MAT + Redox ──
print('Loading SoilGrids...')
sg = pd.read_parquet(ENVDBS / 'SoilGrids' / 'soilgrids_master.parquet',
                     columns=['lat', 'lon', 'pH_10cm', 'soil_organic_carbon_10cm'])
# Note: pH_0-5cm and soil_organic_carbon_0-5cm are all-NaN in this file; use *_10cm columns
sg = sg.rename(columns={'pH_10cm': 'soil_ph', 'soil_organic_carbon_10cm': 'soil_soc'})
sg = sg.dropna(subset=['lat', 'lon'])
print(f'  SoilGrids: {len(sg):,} grid cells, pH notna={sg["soil_ph"].notna().sum():,}')

print('Loading WTD...')
wt = pd.read_parquet(ENVDBS / 'water_table_depth.parquet')
wt = wt.dropna(subset=['lat', 'lon'])
# water_table_depth_m (lowercase) is all-NaN; Water_Table_Depth_m (mixed-case) has values
n_lower = wt['water_table_depth_m'].notna().sum() if 'water_table_depth_m' in wt.columns else 0
n_mixed = wt['Water_Table_Depth_m'].notna().sum() if 'Water_Table_Depth_m' in wt.columns else 0
wtd_col = 'Water_Table_Depth_m' if n_mixed >= n_lower else 'water_table_depth_m'
wt = wt.rename(columns={wtd_col: 'wtd_m'})
print(f'  WTD: {len(wt):,} grid cells, using column={wtd_col}, notna={wt["wtd_m"].notna().sum():,}')

print('Loading WorldClim (MAT)...')
wc = pd.read_parquet(ENVDBS / 'worldclim_master.parquet',
                     columns=['lat', 'lon', 'bio_1', 'bio_12'])
wc = wc.rename(columns={'bio_1': 'wc_mat', 'bio_12': 'wc_map'})
print(f'  WorldClim: {len(wc):,} grid cells')

print('Loading Redox (Wherry et al. 2023 RF predictions)...')
from pyproj import Transformer, CRS
BIOIN_DATA = ROOT / 'projects/metal_contamination_bioindicators/data'
rx_raw = pd.read_parquet(BIOIN_DATA / 'redox10_grid_prediction.parquet',
                         columns=['x_map', 'y_map', 'pred_oxic_5m'])
# Reproject from ESRI:102003 (CONUS Albers) to WGS84 lat/lon
_t = Transformer.from_crs(CRS('ESRI:102003'), CRS('EPSG:4326'), always_xy=True)
_lons, _lats = _t.transform(rx_raw['x_map'].values, rx_raw['y_map'].values)
rx = pd.DataFrame({'lat': _lats, 'lon': _lons, 'p_oxic': rx_raw['pred_oxic_5m'].values})
rx = rx.dropna(subset=['lat', 'lon'])
print(f'  Redox: {len(rx):,} grid cells, p_oxic range=[{rx["p_oxic"].min():.2f}, {rx["p_oxic"].max():.2f}]')

# Build KD-trees for spatial lookups
sg_tree  = cKDTree(sg[['lat', 'lon']].values)
wt_tree  = cKDTree(wt[['lat', 'lon']].values)
wc_tree  = cKDTree(wc[['lat', 'lon']].values)
ug_tree  = cKDTree(usgs_grid[['lat', 'lon']].values)
rx_tree  = cKDTree(rx[['lat', 'lon']].values)
print('KD-trees built.')

Loading SoilGrids...
  SoilGrids: 338,939 grid cells, pH notna=228,205
Loading WTD...
  WTD: 338,939 grid cells, using column=Water_Table_Depth_m, notna=234,165
Loading WorldClim (MAT)...
  WorldClim: 338,939 grid cells
Loading Redox (Wherry et al. 2023 RF predictions)...


  Redox: 2,411,614 grid cells, p_oxic range=[0.02, 0.90]


KD-trees built.


In [4]:
# ── Load MicrobeAtlas USA genus counts → relative abundances ──────────────────
print('Loading MicrobeAtlas USA genus counts...')
otu = pd.read_parquet(USAENV / 'data/nb02_otu_long_cache.parquet')
print(f'  OTU long: {otu.shape}, {otu.sample_id.nunique():,} samples')

# Total count per sample
sample_totals = otu.groupby('sample_id')['count'].sum().rename('total')
otu = otu.merge(sample_totals, on='sample_id', how='left')
otu['rel_abund'] = otu['count'] / otu['total']

# Sample metadata (unique lat/lon per sample_id)
samp_meta = otu[['sample_id', 'lat', 'lon']].drop_duplicates().copy()
print(f'  Unique samples: {len(samp_meta):,}')
print(f'  Unique genera: {otu.genus_lower.nunique():,}')

Loading MicrobeAtlas USA genus counts...
  OTU long: (346716, 8), 6,034 samples
  Unique samples: 6,034
  Unique genera: 200


In [5]:
# ── Build or load sample master table ────────────────────────────────────────
if SAMPLE_MASTER_PATH.exists():
    print('Loading cached sample master...')
    samp_env = pd.read_parquet(SAMPLE_MASTER_PATH)
else:
    print('Building sample master (spatial joins)...')
    samp_ll = samp_meta[['lat', 'lon']].values

    # USGS metals
    ug_dist, ug_idx = ug_tree.query(samp_ll, k=1)
    too_far_ug = ug_dist > MAX_DIST_DEG
    for col in METAL_COLS:
        vals = usgs_grid[col].values[ug_idx].copy().astype(np.float64)
        vals[too_far_ug] = np.nan
        samp_meta[col] = vals
    samp_meta['usgs_dist_deg'] = ug_dist
    n_usgs = (~too_far_ug).sum()
    print(f'  Samples with USGS coverage: {n_usgs:,}/{len(samp_meta):,}')

    # SoilGrids pH + SOC
    _, sg_idx = sg_tree.query(samp_ll, k=1)
    samp_meta['soil_ph']  = sg['soil_ph'].values[sg_idx]
    samp_meta['soil_soc'] = sg['soil_soc'].values[sg_idx]

    # WTD
    _, wt_idx = wt_tree.query(samp_ll, k=1)
    samp_meta['wtd_m'] = wt['wtd_m'].values[wt_idx]

    # WorldClim MAT + MAP
    _, wc_idx = wc_tree.query(samp_ll, k=1)
    samp_meta['wc_mat'] = wc['wc_mat'].values[wc_idx]
    samp_meta['wc_map'] = wc['wc_map'].values[wc_idx]

    # Redox: P(oxic) at 5m depth (Wherry et al. 2023)
    _, rx_idx = rx_tree.query(samp_ll, k=1)
    samp_meta['p_oxic'] = rx['p_oxic'].values[rx_idx]

    samp_meta['lat_abs'] = samp_meta['lat'].abs()
    samp_meta.to_parquet(SAMPLE_MASTER_PATH, index=False)
    samp_env = samp_meta
    print(f'Saved sample master → {SAMPLE_MASTER_PATH.name}')

print(f'Sample master: {samp_env.shape}')
print(f'USGS coverage: {samp_env[METAL_COLS[0]].notna().sum():,} samples for {METAL_COLS[0]}')
print(f'pH coverage: {samp_env["soil_ph"].notna().sum():,}')
print(f'WTD coverage: {samp_env["wtd_m"].notna().sum():,}')
print(f'Redox coverage: {samp_env["p_oxic"].notna().sum():,}')

Loading cached sample master...
Sample master: (6034, 60)
USGS coverage: 1,309 samples for usgs_ag
pH coverage: 6,034
WTD coverage: 6,034
Redox coverage: 6,034


In [6]:
# ── Load KO presence matrix + compute community-weighted KO abundance ─────────
print('Loading KO presence matrix...')
nb25 = pd.read_parquet(DATA / 'nb25_ko_presence_matrix.parquet')
nb25['genus_lower'] = nb25['genus_lower'].str.replace('g__', '', regex=False)

# Load genome size data for KO density calculation
spark_df = pd.read_csv(DATA / '01_genus_ko_density_spark.csv')
nb25 = nb25.merge(spark_df[['genus_lower', 'n_genomes', 'mean_genome_mb']],
                  on='genus_lower', how='inner')
nb25 = nb25[nb25['n_genomes'] >= 2]  # at least 2 genomes
nb25['pres_frac'] = nb25['n_genomes_with_ko'] / nb25['n_genomes'].clip(lower=1)

# Curated MHG KO list — include Tier 1, Tier 2, and Tier 3-BacMet
# Tier 3-BacMet adds canonical specific-resistance KOs (merA/Hg, arsABC/As,
# chrA/Cr, cusC/Cu, aoxA/As) that were missing from the Tier 1+2 subset.
curated = pd.read_csv(DATA / 'curated_mrg_ko_ids_v2.csv')
curated_keep = curated[curated['evidence_tier'].isin(['Tier 1', 'Tier 2', 'Tier 3-BacMet'])]
FITTED_KOS = list(set(curated_keep['KO'].tolist()) & set(nb25['ko'].unique()))
print(f'Tier 1+2+BacMet KOs in nb25: {len(FITTED_KOS)}')
# Show breakdown by tier
for tier in ['Tier 1', 'Tier 2', 'Tier 3-BacMet']:
    n_tier = len(set(curated[curated['evidence_tier'] == tier]['KO']) & set(nb25['ko'].unique()))
    print(f'  {tier}: {n_tier} in nb25')
if len(FITTED_KOS) == 0:
    # Fall back to all metal KOs in nb25
    FITTED_KOS = nb25['ko'].unique().tolist()
    print(f'Using all nb25 KOs: {len(FITTED_KOS)}')

# Community-weighted KO abundance per sample
# cwm(ko, sample) = Σ_g [rel_abund(g, sample) × pres_frac(ko, g)]
print('Computing community-weighted KO abundances...')
ko_pres = nb25[nb25['ko'].isin(FITTED_KOS)][['genus_lower', 'ko', 'pres_frac']].copy()

# Merge OTU abundances with KO presence
otu_ko = otu.merge(ko_pres, on='genus_lower', how='inner')
otu_ko['weighted'] = otu_ko['rel_abund'] * otu_ko['pres_frac']

cwm = (otu_ko.groupby(['sample_id', 'ko'])['weighted']
       .sum()
       .reset_index()
       .rename(columns={'weighted': 'cwm'}))

print(f'CWM: {cwm.shape}, {cwm.sample_id.nunique():,} samples, {cwm.ko.nunique():,} KOs')
# Check coverage
n_genera_linked = otu_ko['genus_lower'].nunique()
n_genera_otu    = otu['genus_lower'].nunique()
print(f'Genera linked to KO data: {n_genera_linked:,}/{n_genera_otu:,} ({100*n_genera_linked/n_genera_otu:.1f}%)')

Loading KO presence matrix...


Tier 1+2+BacMet KOs in nb25: 160
  Tier 1: 16 in nb25
  Tier 2: 29 in nb25
  Tier 3-BacMet: 115 in nb25
Computing community-weighted KO abundances...


CWM: (765683, 3), 6,031 samples, 151 KOs
Genera linked to KO data: 128/200 (64.0%)


In [7]:
# ── Pivot CWM to wide, merge with env, build analysis DataFrame ───────────────
print('Pivoting CWM to wide format...')
cwm_wide = cwm.pivot(index='sample_id', columns='ko', values='cwm').reset_index()
cwm_wide.columns.name = None
ko_cols = [c for c in cwm_wide.columns if c.startswith('K')]

# Merge with env master
df = cwm_wide.merge(samp_env, on='sample_id', how='inner')
print(f'Combined DataFrame: {df.shape}')

# Standardise controls (z-score)
for col in ['soil_ph', 'soil_soc', 'wtd_m', 'wc_mat', 'p_oxic', 'lat_abs']:
    valid = df[col].notna()
    df[f'{col}_z'] = np.nan
    df.loc[valid, f'{col}_z'] = stats.zscore(df.loc[valid, col])

# Log1p transform metals
for mc in METAL_COLS:
    df[f'{mc}_log'] = np.log1p(df[mc])

METAL_LOG_COLS = [f'{mc}_log' for mc in METAL_COLS]

# Standardise KO CWM (z-score per KO)
for ko in ko_cols:
    valid = df[ko].notna()
    df[f'{ko}_z'] = np.nan
    if valid.sum() > 10:
        df.loc[valid, f'{ko}_z'] = stats.zscore(df.loc[valid, ko])

print(f'Samples with ≥1 USGS metal: {df[METAL_COLS[0]].notna().sum():,}')
print(f'Samples with complete core controls: '
      f"{df[['soil_ph_z','soil_soc_z','wtd_m_z']].notna().all(axis=1).sum():,}")
print(f'Samples with redox: {df["p_oxic_z"].notna().sum():,}')

Pivoting CWM to wide format...
Combined DataFrame: (6031, 211)


Samples with ≥1 USGS metal: 1,308
Samples with complete core controls: 6,031
Samples with redox: 6,031


In [8]:
# ── Metal PCA: compute PC1 as geochemical-background control ──────────────────
# USGS metals are strongly correlated (mafic/REE lithology axis, r>0.7 for
# Sc/V/Cr/Ni/Co, REE pairs, Hf/Zr). PC1 explains ~48% of metal variance.
# Any MWAS/PGLS hit that doesn't survive PC1 control is picking up lithology,
# not the specific metal's chemistry.

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Use log1p-transformed metals that are valid for at least 30% of samples
metal_log_notna = df[METAL_LOG_COLS].notna().mean()
pca_metal_cols = [c for c in METAL_LOG_COLS if metal_log_notna[c] >= 0.30]
print(f'Metals entering PCA: {len(pca_metal_cols)} (≥30% sample coverage)')

# Fit PCA on rows where ALL selected metals are non-NaN
pca_mask = df[pca_metal_cols].notna().all(axis=1)
X_pca = df.loc[pca_mask, pca_metal_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)
pca = PCA(n_components=3)
pca.fit(X_scaled)
print(f'PC1 explains {pca.explained_variance_ratio_[0]*100:.1f}% of selected metal variance')
print(f'PC2 explains {pca.explained_variance_ratio_[1]*100:.1f}%')

# Project all rows (fill NaN for rows missing any PCA metal)
X_all = scaler.transform(df[pca_metal_cols].fillna(df[pca_metal_cols].mean()).values)
scores = pca.transform(X_all)
df['metal_pc1'] = np.where(pca_mask, scores[:, 0], np.nan)
df['metal_pc2'] = np.where(pca_mask, scores[:, 1], np.nan)

# Z-score PC1 (it's already centered/scaled but we standardise for consistency)
valid_pc1 = df['metal_pc1'].notna()
df['metal_pc1_z'] = np.nan
df.loc[valid_pc1, 'metal_pc1_z'] = stats.zscore(df.loc[valid_pc1, 'metal_pc1'])
print(f'Samples with metal PC1: {valid_pc1.sum():,}')

# Show top PC1 loadings (for reporting)
load_df = pd.Series(pca.components_[0],
                    index=[c.replace('usgs_', '').replace('_log', '') for c in pca_metal_cols]
                    ).abs().sort_values(ascending=False)
print('Top 8 PC1 loadings (absolute):', load_df.head(8).to_dict())

Metals entering PCA: 40 (≥30% sample coverage)
PC1 explains 89.8% of selected metal variance
PC2 explains 8.6%
Samples with metal PC1: 1,648
Top 8 PC1 loadings (absolute): {'pb': 0.16681323615158095, 'te': 0.16666241447781682, 'ge': 0.1666403224580612, 'cd': 0.16653556022245541, 'zn': 0.1664123297218534, 'se': 0.16637383793404656, 'mo': 0.1662976374104623, 'zr': 0.1662050350609991}


In [9]:
# ── ANALYSIS 1: Sample-level MWAS ─────────────────────────────────────────────
# For each KO × metal × control combination: OLS regression
# metal_log ~ ko_cwm_z + [controls]
#
# Control philosophy:
#   - pH/SOC/WTD/redox/MAT: abiotic confounders that co-vary with metals via soil chemistry
#
# NOTE: metal_pc1 is intentionally EXCLUDED from the MWAS control set.
# PC1 explains ~90% of sample-level metal variance; after removing it, the residual
# metals all reflect the same second latent axis (PC2, 8.6%), making every metal's
# residual proportional to every other metal's residual. This causes identical
# t-statistics across metals for any given KO — producing spurious multi-metal hits
# that reflect one uncontrolled geochemical gradient, not specific metal biology.
# metal_pc1 IS appropriate in the PGLS (genus level, n≈60-110) where genus-level
# metal residuals retain enough independent variation.

CTRL_COMBOS = [
    ('none',                  []),
    ('pH',                    ['soil_ph_z']),
    ('redox',                 ['p_oxic_z']),
    ('pH+redox',              ['soil_ph_z', 'p_oxic_z']),
    ('pH+SOC+WTD',            ['soil_ph_z', 'soil_soc_z', 'wtd_m_z']),
    ('pH+SOC+WTD+redox',      ['soil_ph_z', 'soil_soc_z', 'wtd_m_z', 'p_oxic_z']),
    ('pH+SOC+WTD+redox+MAT',  ['soil_ph_z', 'soil_soc_z', 'wtd_m_z', 'p_oxic_z', 'wc_mat_z']),
]
CTRL_NAMES = [c[0] for c in CTRL_COMBOS]

MIN_N = 50  # minimum samples per regression

print(f'Running MWAS: {len(ko_cols)} KOs × {len(METALS)} metals × {len(CTRL_COMBOS)} combos')
mwas_rows = []

for metal, metal_col_log in zip(METALS, METAL_LOG_COLS):
    metal_valid = df[metal_col_log].notna()
    n_metal = metal_valid.sum()
    if n_metal < MIN_N:
        continue

    for ko in ko_cols:
        ko_z_col = f'{ko}_z'
        if ko_z_col not in df.columns:
            continue

        for ctrl_name, ctrl_cols in CTRL_COMBOS:
            required = [metal_col_log, ko_z_col] + ctrl_cols
            mask = df[required].notna().all(axis=1)
            sub = df.loc[mask].copy()
            if len(sub) < MIN_N:
                continue

            # Standardise metal within this subset
            sub['y'] = stats.zscore(sub[metal_col_log])

            formula_rhs = ' + '.join([ko_z_col] + ctrl_cols)
            try:
                res = smf.ols(f'y ~ {formula_rhs}', data=sub).fit()
                beta  = res.params[ko_z_col]
                se    = res.bse[ko_z_col]
                pval  = res.pvalues[ko_z_col]
                tstat = res.tvalues[ko_z_col]
            except Exception:
                continue

            # Gene info
            ko_info = curated[curated['KO'] == ko].iloc[0] if (curated['KO'] == ko).any() else None
            mwas_rows.append({
                'ko': ko,
                'gene_name': ko_info['gene_name'] if ko_info is not None else '',
                'category':  ko_info['primary_category'] if ko_info is not None else '',
                'metal': metal,
                'ctrl': ctrl_name,
                'n': len(sub),
                'beta': beta,
                'se': se,
                'tstat': tstat,
                'pval': pval,
            })

mwas = pd.DataFrame(mwas_rows)
print(f'MWAS: {len(mwas):,} tests run')

Running MWAS: 151 KOs × 49 metals × 7 combos


MWAS: 50,680 tests run


In [10]:
# ── MWAS: FDR correction per metal × control combo ────────────────────────────
from statsmodels.stats.multitest import multipletests

fdr_rows = []
for (metal, ctrl), grp in mwas.groupby(['metal', 'ctrl']):
    grp = grp.copy()
    reject, pvals_adj, _, _ = multipletests(grp['pval'].values, method='fdr_bh')
    grp['pval_fdr'] = pvals_adj
    grp['sig_fdr']  = reject
    fdr_rows.append(grp)

mwas = pd.concat(fdr_rows, ignore_index=True)

# Summary
summary = (mwas.groupby(['metal', 'ctrl'])['sig_fdr']
           .agg(['sum', 'count'])
           .rename(columns={'sum': 'n_sig', 'count': 'n_tests'})
           .reset_index())
summary['pct_sig'] = 100 * summary['n_sig'] / summary['n_tests']

print('MWAS significant hits (FDR < 5%) by metal × control combo:')
pivot = summary.pivot(index='metal', columns='ctrl', values='n_sig').fillna(0).astype(int)
print(pivot.to_string())

mwas.to_csv(DATA / 'nb33_mwas_results.csv', index=False)
print(f'\nTotal significant hits (any combo): {mwas["sig_fdr"].sum():,}')
print('Top hits:')
print(mwas[mwas['sig_fdr']].sort_values('pval_fdr')
      [['ko','gene_name','category','metal','ctrl','n','beta','pval','pval_fdr']]
      .head(20).to_string())

MWAS significant hits (FDR < 5%) by metal × control combo:
ctrl   none   pH  pH+SOC+WTD  pH+SOC+WTD+redox  pH+SOC+WTD+redox+MAT  pH+redox  redox
metal                                                                                
ag      106   90          91                23                    23        66     79
as      136  142         137               139                   129       143    132
au      138  103          82                74                    50        67    110
b        78   68          65                62                    61        57     60
ba      129  127         102               111                    82       136    129
be      123  110         125               102                   110        92    128
bi      132  130         102                94                    99       102    136
cd      137  135         126               122                   108       123    135
ce      134  114         118               117                    99       120   


Total significant hits (any combo): 34,182
Top hits:
           ko   gene_name                   category metal              ctrl     n      beta  pval  pval_fdr
27274  K16088  TC.FEV.OM1                    Unknown    ni          pH+redox  5882  0.273786   0.0       0.0
21240  K18145        adeA  Resistance/Detoxification    li              none  5802  0.479252   0.0       0.0
26676  K18145        adeA  Resistance/Detoxification    ni                pH  5802  0.474303   0.0       0.0
21234  K16088  TC.FEV.OM1                    Unknown    li              none  5882  0.526339   0.0       0.0
28035  K18145        adeA  Resistance/Detoxification    pb  pH+SOC+WTD+redox  5802 -0.267883   0.0       0.0
10737  K03199       virB4                    Unknown    cr                pH  5966  0.436696   0.0       0.0
28036  K18146        adeB  Resistance/Detoxification    pb  pH+SOC+WTD+redox  5820 -0.242332   0.0       0.0
18063  K16088  TC.FEV.OM1                    Unknown    hg              no

In [11]:
# ── MWAS Figure: N significant hits per control combo ─────────────────────────
# Heatmap: metals (rows) × control combos (cols), colour = n_sig

pivot_sig = summary.pivot(index='metal', columns='ctrl', values='n_sig').fillna(0)
# Reindex to CTRL_NAMES order, keeping only combos that actually ran
present_ctrls = [c for c in CTRL_NAMES if c in pivot_sig.columns]
pivot_sig = pivot_sig.reindex(columns=present_ctrls, fill_value=0)

fig, axs = plt.subplots(1, 2, figsize=(FIGW['full'], ROW_H * 1.4))

# Panel A: heatmap of n_sig
ax = axs[0]
if pivot_sig.shape[1] > 0:
    im = ax.imshow(pivot_sig.values, aspect='auto', cmap='YlOrRd', vmin=0)
    ax.set_xticks(range(len(present_ctrls)))
    ax.set_xticklabels(present_ctrls, rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(len(pivot_sig.index)))
    ax.set_yticklabels(pivot_sig.index, fontsize=7)
    plt.colorbar(im, ax=ax, label='N significant KOs (FDR<5%)', shrink=0.8)
ax.set_xlabel('Control combination')
ax.set_ylabel('USGS element')
ax.set_title('MWAS significant hits by element × control')

# Panel B: volcano plot — use most controlled available combo, fallback to 'none'
ax2 = axs[1]
full_ctrl = next((c for c in ['pH+SOC+WTD', 'pH+SOC', 'pH', 'none'] if c in mwas['ctrl'].values), None)
if full_ctrl:
    mwas_full = mwas[mwas['ctrl'] == full_ctrl].copy()
    ax2.scatter(mwas_full['beta'], -np.log10(mwas_full['pval'].clip(1e-10)),
                s=10, alpha=0.4, color=PALETTE[0], linewidths=0, label='NS')
    sig_full = mwas_full[mwas_full['sig_fdr']]
    if len(sig_full) > 0:
        ax2.scatter(sig_full['beta'], -np.log10(sig_full['pval'].clip(1e-10)),
                    s=20, alpha=0.9, color='firebrick', linewidths=0, label=f'FDR<5% (n={len(sig_full)})')
        # Only label top 10 to avoid overplotting
        for _, row in sig_full.nsmallest(10, 'pval_fdr').iterrows():
            ax2.annotate(f"{row['gene_name']}:{row['metal']}",
                         (row['beta'], -np.log10(row['pval'])),
                         fontsize=6, alpha=0.8)
    ax2.axhline(-np.log10(0.05), color='gray', lw=0.8, ls='--')
    ax2.axvline(0, color='gray', lw=0.8, ls='--')
    ax2.set_xlabel('β (standardised)')
    ax2.set_ylabel('-log₁₀(p)')
    ax2.set_title(f'MWAS volcano plot ({full_ctrl})')
    ax2.legend(fontsize=7)
    grid_h(ax2)

fig.suptitle('NB33 — Sample-level MWAS: community KO abundance vs USGS measured metals', y=1.02)
save(fig, FIGS / 'nb33_mwas_results')
print(f'Figure saved. Volcano control: {full_ctrl}')

Figure saved. Volcano control: pH+SOC+WTD


In [12]:
# ── ANALYSIS 2: Abundance-weighted genus metal exposure ───────────────────────
# For each genus × metal: weighted mean = Σ(rel_abund × metal) / Σ(rel_abund)
# Weight = relative abundance of that genus in each sample

if GENUS_METAL_PATH.exists():
    print('Loading cached genus abundance-weighted metals...')
    genus_aw = pd.read_csv(GENUS_METAL_PATH)
else:
    print('Computing abundance-weighted genus metal exposure...')
    # Join sample raw metal values (not log) + env controls to OTU long
    ctrl_cols_env = [c for c in ['soil_ph', 'soil_soc', 'wtd_m', 'wc_mat', 'p_oxic'] if c in samp_env.columns]
    otu_env = otu.merge(
        samp_env[['sample_id'] + METAL_COLS + ctrl_cols_env],
        on='sample_id', how='inner'
    )
    print(f'  OTU × env merged: {otu_env.shape}')

    # Weighted mean per genus × metal (log1p transform before averaging)
    records = []
    for mc, metal in zip(METAL_COLS, METALS):
        sub = otu_env[otu_env[mc].notna()].copy()
        if len(sub) == 0:
            continue
        sub['metal_log'] = np.log1p(sub[mc])
        aw = (sub.groupby('genus_lower')
              .apply(lambda g: np.average(g['metal_log'], weights=g['rel_abund'])
                     if g['rel_abund'].sum() > 0 else np.nan, include_groups=False)
              .reset_index(name=f'aw_{metal}'))
        records.append(aw)

    # Also compute weighted controls for PGLS covariates
    for ctrl_var in ctrl_cols_env:
        sub = otu_env[otu_env[ctrl_var].notna()].copy()
        if len(sub) == 0:
            continue
        aw_c = (sub.groupby('genus_lower')
                .apply(lambda g: np.average(g[ctrl_var], weights=g['rel_abund'])
                       if g['rel_abund'].sum() > 0 else np.nan, include_groups=False)
                .reset_index(name=f'aw_{ctrl_var}'))
        records.append(aw_c)

    # Merge all per-genus records
    from functools import reduce
    genus_aw = reduce(lambda a, b: a.merge(b, on='genus_lower', how='outer'), records)
    genus_aw.to_csv(GENUS_METAL_PATH, index=False)
    print(f'Saved → {GENUS_METAL_PATH.name}: {genus_aw.shape}')

# Metal columns exclude control variables
_ctrl_aw_cols = {'aw_soil_ph', 'aw_soil_soc', 'aw_wtd_m', 'aw_wc_mat', 'aw_p_oxic'}
AW_METAL_COLS = [c for c in genus_aw.columns if c.startswith('aw_') and c not in _ctrl_aw_cols]
AW_METALS = [c.replace('aw_', '') for c in AW_METAL_COLS]
print(f'Genus-level AW metals: {len(AW_METALS)}')
print(f'Genera with any AW metal: {genus_aw.shape[0]:,}')
print(f'AW control cols available: {[c for c in _ctrl_aw_cols if c in genus_aw.columns]}')

# Summary stats
for col in AW_METAL_COLS[:5]:
    nn = genus_aw[col].notna().sum()
    print(f'  {col}: {nn:,} genera')

Loading cached genus abundance-weighted metals...
Genus-level AW metals: 49
Genera with any AW metal: 200
AW control cols available: ['aw_soil_ph', 'aw_p_oxic', 'aw_wtd_m', 'aw_soil_soc', 'aw_wc_mat']
  aw_ag: 200 genera
  aw_as: 200 genera
  aw_au: 200 genera
  aw_b: 200 genera
  aw_ba: 200 genera


In [13]:
# ── Genus-level metal PCA: aw_metal_pc1 for PGLS control ─────────────────────
# Same logic as sample-level: compute PC1 of all AW metals to capture the
# shared geochemical background at the genus level.

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA as _PCA

# Select AW metal cols with ≥50% genus coverage
aw_notna = genus_aw[AW_METAL_COLS].notna().mean()
pca_aw_cols = [c for c in AW_METAL_COLS if aw_notna[c] >= 0.50]
print(f'AW metals entering genus PCA: {len(pca_aw_cols)} (≥50% genus coverage)')

aw_pca_mask = genus_aw[pca_aw_cols].notna().all(axis=1)
X_aw = genus_aw.loc[aw_pca_mask, pca_aw_cols].values
_sc = StandardScaler()
_pca = _PCA(n_components=3)
_pca.fit(_sc.fit_transform(X_aw))
print(f'Genus metal PC1 explains {_pca.explained_variance_ratio_[0]*100:.1f}% of AW metal variance')

# Project all genera (impute missing with column mean before projecting)
X_aw_all = _sc.transform(genus_aw[pca_aw_cols].fillna(genus_aw[pca_aw_cols].mean()).values)
aw_scores = _pca.transform(X_aw_all)
genus_aw['aw_metal_pc1'] = np.where(aw_pca_mask, aw_scores[:, 0], np.nan)
print(f'Genera with aw_metal_pc1: {genus_aw["aw_metal_pc1"].notna().sum():,}/{len(genus_aw):,}')

AW metals entering genus PCA: 48 (≥50% genus coverage)
Genus metal PC1 explains 42.9% of AW metal variance
Genera with aw_metal_pc1: 195/200


In [14]:
# ── PGLS setup: pre-load tree, define optimised helpers ───────────────────────
from pgls_utils import load_tree, build_vcv, _optimise_lambda, _gls_fit

TREE_PATH = DATA / 'gtdb_bac_genus_pruned.tree'
print('Loading GTDB tree...')
tree = load_tree(str(TREE_PATH))
tree_labels = {t.label.replace(' ', '_').lower() for t in tree.taxon_namespace}
print(f'  Tree loaded: {len(tree_labels):,} taxa')

def zscore_safe(x):
    mu = np.nanmean(x)
    sd = np.nanstd(x, ddof=1)
    return (x - mu) / sd if sd > 0 else x - mu

def fast_pgls_submatrix(V_ko, genera_list, sub_df, y_col, predictor_cols):
    """Run PGLS using a submatrix of a pre-built VCV (avoids tree reload per call).

    V_ko: pre-built VCV for genera_list (n × n).
    genera_list: ordered genus names used to build V_ko (already normalised lowercase).
    sub_df: subset of those genera to fit; must have genus_lower column.
    """
    sub = sub_df.dropna(subset=[y_col] + predictor_cols).copy()
    sub = sub.drop_duplicates('genus_lower')
    if len(sub) < PGLS_MIN_N:
        return None

    # Map genera to indices in V_ko
    norm = {g: g.replace(' ', '_').lower() for g in genera_list}
    idx_map = {norm[g]: i for i, g in enumerate(genera_list)}
    sub['_taxon_low'] = sub['genus_lower'].str.replace(' ', '_').str.lower()
    sub = sub[sub['_taxon_low'].isin(idx_map)]
    if len(sub) < PGLS_MIN_N:
        return None

    idx = [idx_map[t] for t in sub['_taxon_low'].tolist()]
    V_sub = V_ko[np.ix_(idx, idx)]

    y = sub[y_col].values.astype(float)
    X = np.column_stack([np.ones(len(sub))] + [sub[pc].values.astype(float) for pc in predictor_cols])
    try:
        lam, _ = _optimise_lambda(y, X, V_sub)
        ll, sigma2, betas, betas_se, _, _ = _gls_fit(y, X, V_sub, lam)
    except Exception:
        return None

    t_stats = betas / np.where(betas_se > 0, betas_se, np.nan)
    df_resid = len(sub) - len(betas)
    p_values = 2 * stats.t.sf(np.abs(t_stats), df=df_resid)
    n = len(sub)
    return dict(n=n, lambda_est=float(lam),
                beta=float(betas[1]) if len(betas) > 1 else float(betas[0]),
                se=float(betas_se[1]) if len(betas_se) > 1 else float(betas_se[0]),
                p_value=float(p_values[1]) if len(p_values) > 1 else float(p_values[0]))

# Build KO density table (for PGLS predictor)
density = nb25[nb25['ko'].isin(FITTED_KOS)].copy()
density['ko_density'] = density['n_genomes_with_ko'] / (density['n_genomes'] * density['mean_genome_mb'].clip(lower=0.01))

# Filter pgls_base to tree taxa
pgls_base = density.merge(genus_aw, on='genus_lower', how='inner')
pgls_base['_taxon_low'] = pgls_base['genus_lower'].str.replace(' ', '_').str.lower()
pgls_base = pgls_base[pgls_base['_taxon_low'].isin(tree_labels)].copy()
print(f'PGLS base: {pgls_base.shape}, {pgls_base.genus_lower.nunique():,} genera in tree')
print(f'  Control cols in pgls_base: {[c for c in ["aw_soil_ph","aw_soil_soc","aw_wtd_m","aw_wc_mat","aw_p_oxic"] if c in pgls_base.columns]}')

# Check NB27 results for comparison
if (DATA / 'nb27_usgs_pgls_results.csv').exists():
    nb27_res = pd.read_csv(DATA / 'nb27_usgs_pgls_results.csv')
    print(f'NB27 centroid PGLS: {len(nb27_res):,} rows; cols: {list(nb27_res.columns[:8])}')
else:
    nb27_res = None
    print('NB27 PGLS results not found — β comparison skipped')

Loading GTDB tree...
  Tree loaded: 2,283 taxa
PGLS base: (5415, 63), 127 genera in tree
  Control cols in pgls_base: ['aw_soil_ph', 'aw_soil_soc', 'aw_wtd_m', 'aw_wc_mat', 'aw_p_oxic']
NB27 centroid PGLS: 376 rows; cols: ['ko_id', 'metal', 'gene_name', 'subcategory', 'ph_controlled', 'lambda_est', 'beta', 'SE']


In [15]:
# ── PGLS: abundance-weighted metal ~ KO density ───────────────────────────────
# Optimisation: pre-build VCV once per KO (not per KO × metal × ctrl).
# For each metal × ctrl combo: extract submatrix in O(k²) instead of O(n²) MRCA rebuild.
#
# Control philosophy mirrors MWAS:
#   aw_metal_pc1 = genus-level PC1 of all AW metals (lithology/geochemical background)

PGLS_CTRL_COMBOS = [
    ('none',                              []),
    ('pH',                                ['aw_soil_ph']),
    ('redox',                             ['aw_p_oxic']),
    ('pH+redox',                          ['aw_soil_ph', 'aw_p_oxic']),
    ('pH+SOC+WTD',                        ['aw_soil_ph', 'aw_soil_soc', 'aw_wtd_m']),
    ('pH+SOC+WTD+redox',                  ['aw_soil_ph', 'aw_soil_soc', 'aw_wtd_m', 'aw_p_oxic']),
    ('metalPC1',                          ['aw_metal_pc1']),
    ('pH+SOC+WTD+redox+metalPC1',         ['aw_soil_ph', 'aw_soil_soc', 'aw_wtd_m', 'aw_p_oxic', 'aw_metal_pc1']),
    ('pH+SOC+WTD+redox+metalPC1+MAT',     ['aw_soil_ph', 'aw_soil_soc', 'aw_wtd_m', 'aw_p_oxic', 'aw_metal_pc1', 'aw_wc_mat']),
]
PGLS_MIN_N = 25

# Filter combos to available columns, dedup
avail_ctrl = [c for c in ['aw_soil_ph', 'aw_soil_soc', 'aw_wtd_m', 'aw_wc_mat', 'aw_p_oxic', 'aw_metal_pc1']
              if c in pgls_base.columns and pgls_base[c].notna().sum() > 50]
print(f'Available PGLS controls: {avail_ctrl}')
seen_sets = set()
PGLS_CTRL_COMBOS_FINAL = []
for n, cols in PGLS_CTRL_COMBOS:
    filtered = [c for c in cols if c in avail_ctrl]
    key = tuple(sorted(filtered))
    if key not in seen_sets:
        seen_sets.add(key)
        PGLS_CTRL_COMBOS_FINAL.append((n, filtered))
print(f'Control combos after dedup: {[c[0] for c in PGLS_CTRL_COMBOS_FINAL]}')

n_max = len(FITTED_KOS) * len(AW_METALS) * len(PGLS_CTRL_COMBOS_FINAL)
print(f'Max fits: {len(FITTED_KOS)} KOs × {len(AW_METALS)} metals × {len(PGLS_CTRL_COMBOS_FINAL)} combos = {n_max:,}')
pgls_rows = []

for ki, ko in enumerate(FITTED_KOS):
    ko_base = pgls_base[pgls_base['ko'] == ko].drop_duplicates('genus_lower').copy()
    if len(ko_base) < PGLS_MIN_N:
        continue

    # ── Pre-build VCV once for all genera in this KO set ──────────────────────
    genera_list_norm = ko_base['genus_lower'].str.replace(' ', '_').str.lower().tolist()
    try:
        V_ko = build_vcv(tree, genera_list_norm)
    except Exception as e:
        print(f'  VCV build failed for {ko}: {e}')
        continue

    # Standardise KO density
    ko_base = ko_base.copy()
    ko_base['x_std'] = zscore_safe(ko_base['ko_density'].values)

    # Pre-compute control z-scores
    for cc in avail_ctrl:
        if cc in ko_base.columns:
            ko_base[f'{cc}_z'] = zscore_safe(ko_base[cc].values)

    ko_info = curated[curated['KO'] == ko].iloc[0] if (curated['KO'] == ko).any() else None

    # ── Loop over metals × ctrl combos (fast: submatrix extraction only) ──────
    for aw_col, metal in zip(AW_METAL_COLS, AW_METALS):
        if aw_col not in ko_base.columns:
            continue
        # Standardise metal response
        metal_valid = ko_base[aw_col].notna()
        if metal_valid.sum() < PGLS_MIN_N:
            continue
        ko_base.loc[metal_valid, 'y_metal'] = zscore_safe(ko_base.loc[metal_valid, aw_col].values)

        for ctrl_name, ctrl_cols in PGLS_CTRL_COMBOS_FINAL:
            predictor_cols = ['x_std'] + [f'{cc}_z' for cc in ctrl_cols]
            needed = ['y_metal', 'genus_lower'] + predictor_cols
            missing_cols = [c for c in needed if c not in ko_base.columns]
            if missing_cols:
                continue

            res = fast_pgls_submatrix(
                V_ko,
                genera_list_norm,
                ko_base,
                y_col='y_metal',
                predictor_cols=predictor_cols,
            )
            if res is None:
                continue

            pgls_rows.append({
                'ko': ko,
                'gene_name': ko_info['gene_name'] if ko_info is not None else '',
                'category':  ko_info['primary_category'] if ko_info is not None else '',
                'metal': metal,
                'ctrl': ctrl_name,
                'n': res['n'],
                'lambda_est': res['lambda_est'],
                'beta': res['beta'],
                'se': res['se'],
                'p_value': res['p_value'],
                'approach': 'abundance_weighted',
            })

    if (ki + 1) % 5 == 0 or ki == 0:
        print(f'  KO {ki+1}/{len(FITTED_KOS)}: {ko}, rows so far: {len(pgls_rows):,}')

pgls_aw = pd.DataFrame(pgls_rows)
print(f'\nPGLS fits complete: {len(pgls_aw):,} rows')

# FDR per metal × ctrl
from statsmodels.stats.multitest import multipletests
pgls_fdr = []
for (metal, ctrl), grp in pgls_aw.groupby(['metal', 'ctrl']):
    grp = grp.copy()
    reject, padj, _, _ = multipletests(grp['p_value'].values, method='fdr_bh')
    grp['p_fdr'] = padj
    grp['sig_fdr'] = reject
    pgls_fdr.append(grp)

pgls_aw = pd.concat(pgls_fdr, ignore_index=True) if pgls_fdr else pgls_aw
n_sig = pgls_aw['sig_fdr'].sum() if 'sig_fdr' in pgls_aw.columns else 0
print(f'PGLS FDR significant: {n_sig:,}')

pgls_aw.to_csv(DATA / 'nb33_pgls_abund_weighted.csv', index=False)
print('Saved → nb33_pgls_abund_weighted.csv')

Available PGLS controls: ['aw_soil_ph', 'aw_soil_soc', 'aw_wtd_m', 'aw_wc_mat', 'aw_p_oxic', 'aw_metal_pc1']
Control combos after dedup: ['none', 'pH', 'redox', 'pH+redox', 'pH+SOC+WTD', 'pH+SOC+WTD+redox', 'metalPC1', 'pH+SOC+WTD+redox+metalPC1', 'pH+SOC+WTD+redox+metalPC1+MAT']
Max fits: 160 KOs × 49 metals × 9 combos = 70,560


  KO 10/160: K23243, rows so far: 1,296


  KO 20/160: K03543, rows so far: 3,456


  KO 35/160: K17686, rows so far: 6,048


  KO 45/160: K07241, rows so far: 7,776


...

  KO 135/160: K03655, rows so far: 26,352


  KO 140/160: K07789, rows so far: 27,216


  KO 160/160: K04047, rows so far: 30,240

PGLS fits complete: 30,240 rows


PGLS FDR significant: 4,473


Saved → nb33_pgls_abund_weighted.csv


In [16]:
# ── PGLS: Compare abundance-weighted vs NB27 centroid approach ────────────────
if nb27_res is not None and len(nb27_res) > 0:
    print('NB27 columns:', list(nb27_res.columns[:12]))
    # Map NB27 columns: try common patterns
    nb27_metal_col = 'metal' if 'metal' in nb27_res.columns else None
    nb27_ko_col    = 'ko_id' if 'ko_id' in nb27_res.columns else ('ko' if 'ko' in nb27_res.columns else None)
    nb27_beta_col  = 'beta'  if 'beta'  in nb27_res.columns else None
    nb27_ph_col    = 'ph_controlled' if 'ph_controlled' in nb27_res.columns else None
    print(f'  metal={nb27_metal_col}, ko={nb27_ko_col}, beta={nb27_beta_col}, ph_ctrl={nb27_ph_col}')

    if nb27_metal_col and nb27_ko_col and nb27_beta_col:
        # Get 'no pH control' subset from NB27
        if nb27_ph_col:
            nb27_noph = nb27_res[nb27_res[nb27_ph_col] == False].copy()
        else:
            nb27_noph = nb27_res.copy()

        nb27_noph = nb27_noph.rename(columns={
            nb27_beta_col: 'beta_centroid',
            nb27_ko_col:   'ko',
            nb27_metal_col: 'metal',
        })

        # AW PGLS 'none' control
        aw_none = pgls_aw[pgls_aw['ctrl'] == 'none'].copy()
        aw_none = aw_none.rename(columns={'beta': 'beta_aw'})

        # Harmonise metal naming: NB27 may use uppercase (Cu, Pb) vs AW lowercase (cu, pb)
        nb27_noph['metal_lower'] = nb27_noph['metal'].str.lower()
        aw_none['metal_lower']   = aw_none['metal'].str.lower()
        compare = aw_none.merge(nb27_noph[['ko', 'metal_lower', 'beta_centroid']], 
                                on=['ko', 'metal_lower'], how='inner')
        print(f'Overlapping KO × metal pairs: {len(compare)}')
        if len(compare) > 5:
            rho, rho_p = stats.spearmanr(compare['beta_aw'], compare['beta_centroid'])
            print(f'β concordance AW vs centroid (no control): Spearman ρ={rho:.3f}, p={rho_p:.2e}, n={len(compare)}')
        else:
            print('  Too few overlaps for correlation.')
            print('  AW metals (first 5):', sorted(aw_none['metal'].unique())[:5])
            print('  NB27 metals (first 5):', sorted(nb27_noph['metal_lower'].unique())[:5])
    else:
        compare = pd.DataFrame()
        print('Cannot match NB27 columns for comparison.')
else:
    compare = pd.DataFrame()
    print('NB27 results not available — skipping comparison.')

# Summary table per metal × ctrl
if 'sig_fdr' in pgls_aw.columns:
    pgls_summary = (pgls_aw.groupby(['metal', 'ctrl'])
                    .agg(n_sig=('sig_fdr', 'sum'), n_tests=('ko', 'count'))
                    .reset_index())
    pivot_pgls = pgls_summary.pivot(index='metal', columns='ctrl', values='n_sig').fillna(0).astype(int)
    print('\nPGLS significant hits per metal × control:')
    print(pivot_pgls.to_string())

NB27 columns: ['ko_id', 'metal', 'gene_name', 'subcategory', 'ph_controlled', 'lambda_est', 'beta', 'SE', 'p_value', 'n_genera']
  metal=metal, ko=ko_id, beta=beta, ph_ctrl=ph_controlled
Overlapping KO × metal pairs: 126
β concordance AW vs centroid (no control): Spearman ρ=0.319, p=2.74e-04, n=126

PGLS significant hits per metal × control:
ctrl   metalPC1  none  pH  pH+SOC+WTD  pH+SOC+WTD+redox  pH+SOC+WTD+redox+metalPC1  pH+SOC+WTD+redox+metalPC1+MAT  pH+redox  redox
metal                                                                                                                             
ag           34    48  36          38                13                          0                              3        35     53
as            0    38   3           0                 0                          0                              0         0     50
au            0    47   0          23                 1                          0                              0         0     40
b

In [17]:
# ── Figure 2: β control sweep + AW vs centroid β scatter ─────────────────────
fig, axs = plt.subplots(1, 2, figsize=(FIGW['full'], ROW_H))

# Panel A: mean β across control combos for metals that have PGLS results
ax = axs[0]
if len(pgls_aw) > 0:
    # Show all metals with at least one significant hit, else just all metals
    sig_metals = pgls_aw[pgls_aw.get('sig_fdr', pd.Series(False, index=pgls_aw.index))]['metal'].unique() if 'sig_fdr' in pgls_aw.columns else []
    plot_metals = list(sig_metals)[:8] if len(sig_metals) > 0 else sorted(pgls_aw['metal'].unique())[:8]
    ctrl_order  = [c[0] for c in PGLS_CTRL_COMBOS]
    for i, metal in enumerate(plot_metals):
        sub = pgls_aw[pgls_aw['metal'] == metal].groupby('ctrl')['beta'].mean().reindex(ctrl_order)
        if sub.notna().sum() > 0:
            ax.plot(range(len(ctrl_order)), sub.values,
                    marker='o', ms=4, lw=1,
                    color=PALETTE[i % len(PALETTE)], label=metal.upper())
    ax.set_xticks(range(len(ctrl_order)))
    ax.set_xticklabels(ctrl_order, rotation=45, ha='right', fontsize=7)
    ax.axhline(0, color='gray', lw=0.8, ls='--')
    ax.set_xlabel('Control combination')
    ax.set_ylabel('Mean β (KO density ~ metal, PGLS)')
    ax.set_title('PGLS β across control combos (AW approach)')
    if len(plot_metals) > 0:
        ax.legend(fontsize=7, ncol=2)
    grid_h(ax)
else:
    ax.set_title('No PGLS results')
    ax.text(0.5, 0.5, 'Insufficient data', ha='center', va='center', transform=ax.transAxes)

# Panel B: AW vs centroid β scatter
ax2 = axs[1]
if compare is not None and len(compare) > 5:
    ax2.scatter(compare['beta_centroid'], compare['beta_aw'],
                s=15, alpha=0.7, color=PALETTE[0], edgecolors='k', linewidths=0.3)
    all_betas = pd.concat([compare['beta_centroid'], compare['beta_aw']]).abs()
    lim = all_betas.max() * 1.1
    ax2.set_xlim(-lim, lim)
    ax2.set_ylim(-lim, lim)
    ax2.axline((0, 0), slope=1, color='gray', lw=0.8, ls='--')
    ax2.axhline(0, color='gray', lw=0.5, ls=':')
    ax2.axvline(0, color='gray', lw=0.5, ls=':')
    rho, _ = stats.spearmanr(compare['beta_centroid'], compare['beta_aw'])
    ax2.annotate(f'ρ={rho:.2f}, n={len(compare)}', xy=(0.05, 0.95), xycoords='axes fraction',
                 va='top', fontsize=9)
    ax2.set_xlabel('β centroid (NB27)')
    ax2.set_ylabel('β abundance-weighted (NB33)')
    ax2.set_title('β concordance: centroid vs abundance-weighted')
else:
    ax2.set_title('NB27 comparison not available')
    ax2.text(0.5, 0.5, 'No NB27 overlap\n(different metal sets or KO sets)',
             ha='center', va='center', transform=ax2.transAxes, fontsize=9)
    ax2.set_xlabel('β centroid (NB27)')
    ax2.set_ylabel('β abundance-weighted (NB33)')

fig.suptitle('NB33 — Abundance-weighted PGLS: control sweep + NB27 comparison', y=1.02)
save(fig, FIGS / 'nb33_pgls_comparison')
print('Figure 2 saved.')

Figure 2 saved.


In [18]:
# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 70)
print('NB33 SUMMARY')
print('=' * 70)
print(f'MicrobeAtlas USA samples: {df.shape[0]:,}')
print(f'KOs tested (CWM): {len([c for c in ko_cols if f"{c}_z" in df.columns]):,}')
print(f'USGS metals: {len(METALS)}')
print(f'Control combinations tested (MWAS): {len(CTRL_COMBOS)}')
print()
print('MWAS:')
print(f'  Total tests: {len(mwas):,}')
n_sig_mwas = mwas['sig_fdr'].sum() if 'sig_fdr' in mwas.columns else 0
print(f'  FDR-significant: {n_sig_mwas:,}')
if 'sig_fdr' in mwas.columns and n_sig_mwas > 0:
    # Full kitchen-sink control
    ks_ctrl = 'pH+SOC+WTD+redox+MAT'
    ks_sig = mwas[(mwas['ctrl'] == ks_ctrl) & mwas['sig_fdr']] if ks_ctrl in mwas['ctrl'].values else pd.DataFrame()
    print(f'  Kitchen-sink ({ks_ctrl}) FDR-sig: {len(ks_sig):,}')
    top = mwas[mwas['sig_fdr']].sort_values('pval_fdr').head(5)
    for _, r in top.iterrows():
        print(f'    {r["gene_name"]}×{r["metal"]} ({r["ctrl"]}): β={r["beta"]:+.3f}, FDR={r["pval_fdr"]:.2e}')
print()
print('Abundance-weighted PGLS:')
print(f'  Total fits: {len(pgls_aw):,}')
if 'sig_fdr' in pgls_aw.columns:
    n_sig_pgls = pgls_aw['sig_fdr'].sum()
    print(f'  FDR-significant (any ctrl): {n_sig_pgls:,}')
    ks_pgls = pgls_aw[pgls_aw['ctrl'] == 'pH+SOC+WTD+redox+metalPC1+MAT']
    if 'sig_fdr' in ks_pgls.columns:
        print(f'  Kitchen-sink (pH+SOC+WTD+redox+metalPC1+MAT) FDR-sig: {ks_pgls["sig_fdr"].sum():,}')
        top_pgls = ks_pgls[ks_pgls['sig_fdr']].sort_values('p_fdr')
        for _, r in top_pgls.iterrows():
            print(f'    {r["gene_name"]}×{r["metal"]}: β={r["beta"]:+.3f}, λ={r["lambda_est"]:.2f}, FDR={r["p_fdr"]:.3f}')
print()
print('Key design notes:')
print('  MWAS controls: none/pH/redox/pH+redox/pH+SOC+WTD/pH+SOC+WTD+redox/pH+SOC+WTD+redox+MAT')
print('  metalPC1 excluded from MWAS (explains 90% of sample-level metal variance;')
print('  residuals collapse to identical t-stats across all metals for each KO).')
print('  metalPC1 retained in PGLS (genus level: enough independent variation remains).')
print('  NB27: genus centroid → USGS within 1.5° → mean metal → PGLS')
print('  NB33 PGLS: sample-level abundance-weighted metal → genus exposure → PGLS')

NB33 SUMMARY
MicrobeAtlas USA samples: 6,031
KOs tested (CWM): 151
USGS metals: 49
Control combinations tested (MWAS): 7

MWAS:
  Total tests: 50,680
  FDR-significant: 34,182
  Kitchen-sink (pH+SOC+WTD+redox+MAT) FDR-sig: 3,916
    TC.FEV.OM1×ni (pH+redox): β=+0.274, FDR=0.00e+00
    adeA×li (none): β=+0.479, FDR=0.00e+00
    adeA×ni (pH): β=+0.474, FDR=0.00e+00
    TC.FEV.OM1×li (none): β=+0.526, FDR=0.00e+00
    adeA×pb (pH+SOC+WTD+redox): β=-0.268, FDR=0.00e+00

Abundance-weighted PGLS:
  Total fits: 30,240
  FDR-significant (any ctrl): 4,473
  Kitchen-sink (pH+SOC+WTD+redox+metalPC1+MAT) FDR-sig: 11
    gadC×cs: β=-0.707, λ=1.00, FDR=0.004
    hoxN×cs: β=-0.532, λ=0.80, FDR=0.004
    gadC×eu: β=-0.543, λ=0.12, FDR=0.008
    hoxN×eu: β=-0.422, λ=0.00, FDR=0.008
    cusC×tl: β=+0.102, λ=0.00, FDR=0.021
    mdtC×zr: β=+0.292, λ=0.00, FDR=0.036
    cobN×ag: β=-0.307, λ=0.00, FDR=0.037
    cusR×ce: β=+0.055, λ=1.00, FDR=0.039
    emrA×tl: β=+0.067, λ=0.00, FDR=0.039
    nikR×ag: β=-0.2